# 05 · Published out-of-fold members: verify, then gate

**Role in the stack:** the three largest weights. `ridge` (heuljax generator-aware ridge LR) weight **0.386**, `heuljax_s42`
(his plain LR) 0.204, `aryan_s42` (HistGradientBoosting) 0.147. Added in the final 48 hours; together +0.00036 CV, +0.00024 public LB.

## The rule
A public notebook's OOF file is only usable if (a) it was built on **our exact folds** (StratifiedKFold 5, shuffle, seed 42 is the
community contract this episode), or it is honestly cross-validated and we accept a small optimism; (b) its ids align with our
train/test; (c) its labels match ours; (d) it passes the same paired gate as anything we trained ourselves: stack with it vs stack
without it, mean gain > 2 sd across the 5 folds, every fold up.

Files come from `kaggle kernels output <user>/<slug>`. Nothing is re-trained.

| member | author, notebook | author's CV | on our y | corr with our CatBoost | stack gain |
|---|---|---|---|---|---|
| `heuljax` | Paul Bryan Elefante, *KPS6E09 Logistic Regression Sample* | 0.94626 (10-fold) | 0.94626 | 0.948 | +0.00018, sd 0.00004, 5/5 |
| `aryan` | Aryan Kaisth, *Signal that Matters (HistGBM)* | 0.94615 (20-fold) | 0.94615 | 0.998 | +0.00002, sd 0.00001, 5/5 (at the noise floor) |
| `ridge` | Paul Bryan Elefante, *Generator-Aware Ridge Logistic Regression* | 0.94640 (10-fold) | 0.94640 | 0.855 | **+0.00017, sd 0.00003, 5/5** |

The ridge LR's 0.855 correlation with our CatBoost family is the lowest of any strong member we ever found. Its features are
target rates keyed by the **GPT-2 BPE tokens of the income string** (" 50000" -> `G5|0000`, " 75000" -> `G75|000`), a signal
no decimal binning sees. Every top-15 write-up names this feature family as its largest late gain.

In [ ]:
import sys, numpy as np, pandas as pd
sys.path.insert(0, '../src')
from cv import load, run_cv, paired, SEED, NFOLDS, OOF, TEST
X, y, Xte, te_id = load()          # data/train.csv, data/test.csv  (kaggle competitions download -c playground-series-s6e9)
print(X.shape, Xte.shape, 'positives %.4f' % y.mean())

In [ ]:
import glob
from scipy.special import logit, expit
from sklearn.metrics import roc_auc_score
L = lambda p: logit(np.clip(p, 1e-6, 1 - 1e-6))
tr_id = pd.read_csv('../data/train.csv', usecols=['id'])['id'].values

# 1. align by id and verify labels. Example: the ridge notebook's parquet outputs.
ho = pd.read_parquet(glob.glob('../research/heuljax_ridge/out/oof/*.parquet')[0]).set_index('id')
ht = pd.read_parquet(glob.glob('../research/heuljax_ridge/out/test_preds/*.parquet')[0]).set_index('id')
h_oof, h_test = ho.loc[tr_id, 'oof_pred'].values, ht.loc[te_id, [c for c in ht.columns if 'pred' in c][0]].values
assert np.array_equal(ho.loc[tr_id, 'y_true'].values, y.values), 'labels do not match -> wrong folds or wrong rows'
print('single on our y: %.5f' % roc_auc_score(y, h_oof))
paired(h_oof, np.load(OOF / 'oof_final.npy'), y, 'published LR vs our additive tree')

In [ ]:
# 2. gate: stack WITH vs stack WITHOUT (see notebook 06 for fit_stack). Only --write on PASS.
# py src/stack_check_published_oof.py            -> prints the paired comparison
# py src/stack_check_published_oof.py --write     -> saves outputs/oof/oof_ridge_s42.npy, outputs/test/test_ridge_s42_full.npy
#                                                    and submissions/sub_ridge_stack_cv<cv>.csv

## What we learned
- A borrowed OOF column is the cheapest member you will ever add, and the gate catches the useless ones: najiama's own published
  OOF blend was +0.00002 (we already had the recipe), the jazivxt "Zoom Zoom" port +0.00002 (corr 0.998 with naji).
- The mistake we made: we treated heuljax's notebooks as columns to borrow, one day before the deadline. The teams that finished
  top 10 ported his **features** onto their folds and trained residual GBDTs on them (LR logit as `base_margin`). That is the
  0.00016 between our 192nd and 10th place.
- A 10-fold OOF reads ~0.0001 optimistic next to 5-fold members. Our CV-to-LB offset shrank from +0.00027 to +0.00015 when these
  entered, consistent with that.